# Analytics Store Tutorial

A guide to saving and querying capability results with the `checkmaite` analytics store.

> **NOTE:** This tutorial demonstrates the analytics store using object detection (OD) datasets and the `DataevalCleaning` capability. The analytics store works identically for image classification (IC) capabilities.

## What is the Analytics Store?

When you run a capability (such as `DataevalCleaning` or `DataevalBias`), the results are Python objects in memory. They disappear when your notebook restarts. The **analytics store** solves this by:

- **Persisting results** via a pluggable storage backend ([Parquet](https://parquet.apache.org/) by default) so they survive notebook restarts
- **Enabling SQL queries** across runs, datasets, and capabilities
- **Supporting external tools** — with the default Parquet backend, DuckDB, Spark, pandas, or any Parquet reader can query the same files

Each capability extracts scalar summary metrics from its rich outputs (e.g., duplicate count, outlier ratio) into flat records that map directly to SQL rows.

The store manages two kinds of tables:
- **Capability tables** (e.g., `dataeval_cleaning`) — one per capability, containing the extracted metrics
- **`runs` table** — auto-populated metadata mapping each run to its datasets, models, and metrics

## Setup

In [ ]:
import tempfile
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core.object_detection import DataevalCleaning
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset

# Create a store backed by a temporary directory
# In practice, use a persistent path like "./analytics_store"
store_dir = tempfile.mkdtemp(prefix="analytics_store_guide_")
store = AnalyticsStore(ParquetBackend(store_dir))
print(f"Store created at: {store_dir}")

## Running a Capability and Saving Results

The typical workflow is: load a dataset, run a capability, then write the results to the store.

We use the `CocoDetectionDataset` wrapper to load a small subset of the COCO 2017 dataset included in the repository's test data.

In [ ]:
# Load the example COCO dataset
BASE_DIR = Path.cwd().parents[1]
dataset_root = BASE_DIR / "tests/data_for_tests/coco_dataset"
dataset_ann = dataset_root / "ann_file.json"

dataset = CocoDetectionDataset(
    root=str(dataset_root),
    ann_file=str(dataset_ann),
    dataset_id="coco-example",
)
print(f"Loaded dataset: {dataset.metadata['id']} ({len(dataset)} images)")

# Run DataevalCleaning
capability = DataevalCleaning()
run = capability.run(datasets=[dataset], use_cache=False)

# Write to the store
store.write([run])

# Inspect what was written
# list_tables() shows which capabilities have written results
print(f"\nTables: {store.list_tables()}")

# describe_table() shows the available columns and types — useful for writing SQL queries
print(f"\nSchema for 'dataeval_cleaning':")
for col, dtype in store.describe_table("dataeval_cleaning").items():
    print(f"  {col}: {dtype}")

## Querying Results via SQL

The store exposes a `query_sql()` method that accepts standard SQL and returns a [Polars](https://docs.pola.rs/) DataFrame.

In [ ]:
# View all cleaning records
df = store.query_sql("SELECT * FROM dataeval_cleaning")
print(df)

In [ ]:
# Select specific fields
df = store.query_sql("""
    SELECT
        dataset_id,
        exact_duplicate_count,
        exact_duplicate_ratio,
        image_outlier_count,
        image_outlier_ratio,
        mean_brightness
    FROM dataeval_cleaning
""")
print(df)

## Comparing Across Runs

Every run gets a unique `run_uid` and the store deduplicates by this key. You can run the same capability on different datasets and compare results side by side.

The store also auto-populates a `runs` table that maps each `run_uid` to its datasets, models, and metrics.

In [ ]:
# Run cleaning on a second dataset
dataset_root_2 = BASE_DIR / "tests/data_for_tests/coco_resized_val2017"
dataset_ann_2 = dataset_root_2 / "instances_val2017_resized_6.json"

dataset_2 = CocoDetectionDataset(
    root=str(dataset_root_2),
    ann_file=str(dataset_ann_2),
    dataset_id="coco-resized",
)

run_2 = capability.run(datasets=[dataset_2], use_cache=False)
store.write([run_2])

# Compare cleaning results across both datasets
df = store.query_sql("""
    SELECT
        dataset_id,
        exact_duplicate_count,
        image_outlier_count,
        image_outlier_ratio,
        mean_brightness
    FROM dataeval_cleaning
    ORDER BY dataset_id
""")
print(df)

In [ ]:
# Check the auto-populated runs table
df_runs = store.query_sql("""
    SELECT run_uid, capability_table, entity_type, entity_id
    FROM runs
    ORDER BY entity_id
""")
print(df_runs)

## Using Results Outside Python

With the default `ParquetBackend`, the store writes plain Parquet files with scalar columns only. Any tool that reads Parquet can query them directly — no Python required.

**DuckDB example** (run from any SQL client):

```sql
SELECT dataset_id, exact_duplicate_ratio, image_outlier_ratio
FROM read_parquet('./analytics_store/dataeval_cleaning/*.parquet')
ORDER BY image_outlier_ratio DESC;
```

The file layout on disk:

```
analytics_store/
  dataeval_cleaning/
    1706000000000_a1b2c3d4.parquet
  runs/
    1706000000000_e5f6a7b8.parquet
```

Each `write()` call creates one Parquet file per table. File names are `{timestamp_ms}_{uuid}.parquet`.

In [ ]:
# Show actual files on disk
for p in sorted(Path(store_dir).rglob("*.parquet")):
    print(f"  {p.relative_to(store_dir)}  ({p.stat().st_size:,} bytes)")

## Next Steps

In this guide you learned how to:
- Create an analytics store with a storage backend (Parquet by default)
- Run a capability and write its results to the store
- Query results via SQL and compare across datasets
- Access the stored data from external tools (e.g., DuckDB, Spark)

To learn more:
- Run other capabilities (`DataevalBias`, `MaiteEvaluation`, etc.) and write their results to the same store for cross-capability SQL JOINs via `dataset_id`
- For developer documentation on implementing `extract()` for new capabilities, see the [Key Concepts](../development/key_concepts.md) page